
# 99 - Rideshare Project Cleanup and Reset

Use this notebook to reset your work if something goes wrong.
All cleanup actions are **off by default** — change the flag to `True` only when needed.

| Level | What it does | When to use |
|-------|-------------|-------------|
| 1 | Clear processed output files | Your transformation results are wrong, and you want to redo them |
| 2 | Clear source landing files | You copied source files incorrectly |
| 3 | Full project teardown | You want to start over from scratch |

> **Important:** Level 3 removes everything (catalog, schemas, volumes,
> external location, and the ADLS folder). The storage credential
> `ac_dev_dbx_eus2` is never removed.
>
> Run **only the cell you need** — do not run all cells at once.

In [0]:
# This function deletes all files and folders inside a volume path.
# It does NOT delete the volume itself — only its contents.

def clear_volume_contents(volume_path: str) -> None:
    try:
        items = dbutils.fs.ls(volume_path)
    except Exception:
        print(f"  Volume not found or empty: {volume_path}")
        return

    for item in items:
        dbutils.fs.rm(item.path, recurse=True)

    print(f"  Cleared: {volume_path}")


---
### ⚠️ Cleanup Options

**How to use:**
1. Run **Cell 2** first (it loads the helper function)
2. Find the level that matches your problem
3. Change the flag from `False` to `True`
4. Run **only that cell** (not the whole notebook)
5. Change the flag back to `False` when done

> If you already ran this once and some steps show "not found", that's
> normal — it means those items were already removed.

In [0]:
# -------------------------------------------------------
# Level 1 — Delete processed output files
#
# What happens when True:
#   - Deletes all output folders (KPI results) inside the processed volume
#   - The volume and schema stay in place
#
# What happens when False (default):
#   - Nothing is deleted
# -------------------------------------------------------

RESET_PROCESSED_OUTPUTS = False  # ← Change to True, then run this cell

processed_volume = "/Volumes/rideshare_dev/processed/output_files"

if RESET_PROCESSED_OUTPUTS:
    clear_volume_contents(processed_volume)
    print("\n✓ Done. Re-run your transformation cells to regenerate outputs.")
    print("\nRemember: set RESET_PROCESSED_OUTPUTS back to False.")
else:
    print("Skipped (RESET_PROCESSED_OUTPUTS = False).")
    print(f"  Would clear: {processed_volume}")

In [0]:
# -------------------------------------------------------
# Level 2 — Delete source landing files
#
# What happens when True:
#   - Deletes all dataset folders inside the source volume
#   - The volume and schema stay in place
#
# What happens when False (default):
#   - Nothing is deleted
# -------------------------------------------------------

RESET_SOURCE_FILES = False  # ← Change to True, then run this cell

source_volume = "/Volumes/rideshare_dev/landing/source_files"

if RESET_SOURCE_FILES:
    clear_volume_contents(source_volume)
    print("\n✓ Done. Re-run folder creation and file copy cells in Notebook 01.")
    print("\nRemember: set RESET_SOURCE_FILES back to False.")
else:
    print("Skipped (RESET_SOURCE_FILES = False).")
    print(f"  Would clear: {source_volume}")

In [0]:
# -------------------------------------------------------
# Level 3 — Full project teardown (start over from scratch)
#
# What happens when True:
#   1. Delete files inside external volumes
#   2. Drop the rideshare_dev catalog (and everything in it)
#   3. Drop the el_rideshare_dev external location
#   4. Delete the rideshare/ folder from ADLS storage
#
# What is NOT touched:
#   - Storage credential ac_dev_dbx_eus2 (stays for reuse)
#
# What happens when False (default):
#   - Nothing is dropped or deleted
# -------------------------------------------------------

FULL_PROJECT_TEARDOWN = False  # ← Change to True, then run this cell

source_volume = "/Volumes/rideshare_dev/landing/source_files"
processed_volume = "/Volumes/rideshare_dev/processed/output_files"
adls_path = "abfss://container-dev-dbx@sadevdbxeus2.dfs.core.windows.net/rideshare"

if FULL_PROJECT_TEARDOWN:
    # Step 1: Delete files inside external volumes
    print("Step 1: Clearing volume files...")
    for vol in [source_volume, processed_volume]:
        try:
            clear_volume_contents(vol)
        except Exception as e:
            print(f"  Could not clear {vol}: {e}")

    # Step 2: Drop the catalog (removes all schemas, tables, volumes)
    print("\nStep 2: Dropping rideshare_dev catalog...")
    spark.sql("DROP CATALOG IF EXISTS rideshare_dev CASCADE")
    print("  Done.")

    # Step 3: Drop the external location
    print("\nStep 3: Dropping external location...")
    spark.sql("DROP EXTERNAL LOCATION IF EXISTS el_rideshare_dev FORCE")
    print("  Done.")

    # Step 4: Delete the ADLS folder (physical storage cleanup)
    # Note: Unity Catalog may block this if it still detects a path overlap.
    # If that happens, delete the folder manually from Azure Portal.
    print("\nStep 4: Removing rideshare/ folder from ADLS...")
    try:
        dbutils.fs.rm(adls_path, recurse=True)
        print("  Done.")
    except Exception as e:
        if "LOCATION_OVERLAP" in str(e):
            print("  Could not delete — Unity Catalog still protects this path.")
            print("  Manual step: Go to Azure Portal > Storage Account > Containers")
            print("  > container-dev-dbx > select 'rideshare' folder > Delete")
        else:
            print(f"  Error: {e}")

    print("\n✓ Teardown complete. Run Notebook 01 from the top to rebuild.")
    print("\nRemember: set FULL_PROJECT_TEARDOWN back to False.")
else:
    print("Skipped (FULL_PROJECT_TEARDOWN = False).")
    print("  Would drop: rideshare_dev catalog")
    print("  Would drop: el_rideshare_dev external location")
    print("  Would delete: rideshare/ folder in ADLS")
    print("  Would NOT touch: ac_dev_dbx_eus2 storage credential")